# NOTEBOOK C
___

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [2]:
# DATASET LOAD & OVERVIEW
path = 'SRT2026.csv'
df = pd.read_csv(path)
print(df.info())
print(df.isnull().sum())
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 8273 entries, 0 to 8272
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                8273 non-null   str    
 1   LEVEL             8273 non-null   int64  
 2   TYPE              8273 non-null   str    
 3   DEPARTMENT        8273 non-null   str    
 4   DOB               8273 non-null   str    
 5   AGE               8273 non-null   float64
 6   GENERAL           8273 non-null   int64  
 7   TECHNICAL         8273 non-null   int64  
 8   DIGITAL           8273 non-null   int64  
 9   TOTALTRAINING     8273 non-null   int64  
 10  TURN60            8273 non-null   str    
 11  RETIREMENTYEAR    8273 non-null   int64  
 12  RETIREMENTDATE    8273 non-null   str    
 13  RETIREGROUP       8273 non-null   str    
 14  YEARS2RETIREMENT  8273 non-null   float64
dtypes: float64(2), int64(6), str(7)
memory usage: 969.6 KB
None
ID                  0
LEVEL             

,ID,LEVEL,TYPE,DEPARTMENT,DOB,AGE,GENERAL,TECHNICAL,DIGITAL,TOTALTRAINING,TURN60,RETIREMENTYEAR,RETIREMENTDATE,RETIREGROUP,YEARS2RETIREMENT
0,SRT_1,14,Governance & Support,Governor Bureau,1971-09-08,54.32,1,0,0,1,2031-09-08,2031,2031-10-01,A (JAN-SEP),5.75
1,SRT_2,14,Governance & Support,Governor Bureau,1965-10-11,60.22,1,0,0,1,2025-10-11,2026,2026-10-01,B (OCT-DEC),0.75
2,SRT_3,14,Governance & Support,Governor Bureau,1970-08-18,55.37,0,0,0,0,2030-08-18,2030,2030-10-01,A (JAN-SEP),4.75
3,SRT_4,14,Governance & Support,Governor Bureau,1967-07-02,58.50,0,0,0,0,2027-07-02,2027,2027-10-01,A (JAN-SEP),1.75
4,SRT_5,14,Governance & Support,Governor Bureau,1969-06-19,56.54,0,0,0,0,2029-06-19,2029,2029-10-01,A (JAN-SEP),3.75


In [3]:
# 0. PREPREPATION
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
df['STATUS'] = (df['TOTALTRAINING'] > 0).astype(int) #BINARY LABELLING
print(f'NUMBER OF COLUMNS: {df.shape[1]}')
df.head()

NUMBER OF COLUMNS: 16


,ID,LEVEL,TYPE,DEPARTMENT,DOB,AGE,GENERAL,TECHNICAL,DIGITAL,TOTALTRAINING,TURN60,RETIREMENTYEAR,RETIREMENTDATE,RETIREGROUP,YEARS2RETIREMENT,STATUS
0,SRT_1,14,Governance & Support,Governor Bureau,1971-09-08,54.3200,1,0,0,1,2031-09-08,2031,2031-10-01,A (JAN-SEP),5.7500,1
1,SRT_2,14,Governance & Support,Governor Bureau,1965-10-11,60.2200,1,0,0,1,2025-10-11,2026,2026-10-01,B (OCT-DEC),0.7500,1
2,SRT_3,14,Governance & Support,Governor Bureau,1970-08-18,55.3700,0,0,0,0,2030-08-18,2030,2030-10-01,A (JAN-SEP),4.7500,0
3,SRT_4,14,Governance & Support,Governor Bureau,1967-07-02,58.5000,0,0,0,0,2027-07-02,2027,2027-10-01,A (JAN-SEP),1.7500,0
4,SRT_5,14,Governance & Support,Governor Bureau,1969-06-19,56.5400,0,0,0,0,2029-06-19,2029,2029-10-01,A (JAN-SEP),3.7500,0


___
Age Differences Across Groups

In [4]:
# 1.1 AGE DIFFERENCE: TRAINED vs UNTRAINED (INDEPENDENT T-TEST)
untrained = df[df['STATUS'] == 0]['AGE']
trained = df[df['STATUS'] == 1]['AGE']
t_stat, p_val = stats.ttest_ind(trained, untrained, equal_var=False)

print('=' * 40)
print('T-TEST (AGE VS TRAINING STATUS)')
print('=' * 40)
pd.DataFrame({
    'METRIC': ['MEAN AGE(NO TRAINING)', 'MEAN AGE (WITH TRAINING)', 'T-STATISTIC', 'P-VALUE'],
    'VALUE': [untrained.mean(), trained.mean(), t_stat, p_val]
    })

T-TEST (AGE VS TRAINING STATUS)


,METRIC,VALUE
0,MEAN AGE(NO TRAINING),47.6496
1,MEAN AGE (WITH TRAINING),44.0365
2,T-STATISTIC,-16.9516
3,P-VALUE,0.0000


In [5]:
# 1.2 AGE ACROSS TYPE (ANOVA)
model = smf.ols('AGE ~ C(TYPE)', data=df).fit()
anovatable = sm.stats.anova_lm(model, typ=2)

f_stat = anovatable.loc['C(TYPE)', 'F']
p_val = anovatable.loc['C(TYPE)', 'PR(>F)']
gmeans = df.groupby('TYPE')['AGE'].mean()

print('=' * 40)
print('ANOVA (AGE VS TYPE)')
print('=' * 40)
etasq = anovatable.loc['C(TYPE)', 'sum_sq'] / anovatable['sum_sq'].sum()
print(f'Effect Size (η²): {etasq:.4f}')
pd.DataFrame({
    'METRIC': [*[f'MEAN AGE ({x})' for x in gmeans.index],'F-STATISTIC', 'P-VALUE'],
    'VALUE': [*gmeans.values, f_stat, p_val]
})

ANOVA (AGE VS TYPE)
Effect Size (η²): 0.0018


,METRIC,VALUE
0,MEAN AGE (Governance & Support),45.3622
1,MEAN AGE (Infrastructure & Rolling Stock),45.1325
2,MEAN AGE (Operations),45.8663
3,MEAN AGE (Strategic Business & Real Estate),43.0977
4,F-STATISTIC,4.8464
5,P-VALUE,0.0023


In [6]:
# 1.3 AGE ACROSS LEVEL (ANOVA)
model = smf.ols('AGE ~ C(LEVEL)', data=df).fit()
anovatable = sm.stats.anova_lm(model, typ=2)

f_stat = anovatable.loc['C(LEVEL)', 'F']
p_val = anovatable.loc['C(LEVEL)', 'PR(>F)']
gmeans = df.groupby('LEVEL')['AGE'].mean()

print('=' * 40)
print('ANOVA (AGE VS LEVEL)')
print('=' * 40)
etasq = anovatable.loc['C(LEVEL)', 'sum_sq'] / anovatable['sum_sq'].sum()
print(f'Effect Size (η²): {etasq:.4f}')
pd.DataFrame({
    'METRIC': [*[f'MEAN AGE (LEVEL {x})' for x in gmeans.index],'F-STATISTIC', 'P-VALUE'],
    'VALUE': [*gmeans.values, f_stat, p_val]
})

ANOVA (AGE VS LEVEL)
Effect Size (η²): 0.0928


,METRIC,VALUE
0,MEAN AGE (LEVEL 1),53.9460
1,MEAN AGE (LEVEL 2),48.9572
2,MEAN AGE (LEVEL 3),43.1727
3,MEAN AGE (LEVEL 4),46.9397
4,MEAN AGE (LEVEL 5),41.4848
5,MEAN AGE (LEVEL 6),44.5193
6,MEAN AGE (LEVEL 7),50.7477
7,MEAN AGE (LEVEL 8),48.7240
8,MEAN AGE (LEVEL 10),50.5294
9,MEAN AGE (LEVEL 11),50.7866


___
Training Intensity Differences

In [7]:
# 2.1 TOTALTRAINING ACROSS TYPE (ANOVA)
model = smf.ols('TOTALTRAINING ~ C(TYPE)', data=df).fit()
anovatable = sm.stats.anova_lm(model, typ=2)

f_stat = anovatable.loc['C(TYPE)', 'F']
p_val = anovatable.loc['C(TYPE)', 'PR(>F)']
gmeans = df.groupby('TYPE')['TOTALTRAINING'].mean()

print('=' * 40)
print('ANOVA (TOTALTRAINING VS TYPE)')
print('=' * 40)
etasq = anovatable.loc['C(TYPE)', 'sum_sq'] / anovatable['sum_sq'].sum()
print(f'Effect Size (η²): {etasq:.4f}')
pd.DataFrame({
    'METRIC': [*[f'MEAN TRAINING ({x})' for x in gmeans.index], 'F-STATISTIC', 'P-VALUE'],
    'VALUE': [*gmeans.values, f_stat, p_val
    ]
})

ANOVA (TOTALTRAINING VS TYPE)
Effect Size (η²): 0.1778


,METRIC,VALUE
0,MEAN TRAINING (Governance & Support),5.9499
1,MEAN TRAINING (Infrastructure & Rolling Stock),1.7101
2,MEAN TRAINING (Operations),1.3387
3,MEAN TRAINING (Strategic Business & Real Estate),8.7231
4,F-STATISTIC,595.8795
5,P-VALUE,0.0000


In [8]:
# 2.2 TOTALTRAINING ACROSS LEVEL (ANOVA)
model = smf.ols('TOTALTRAINING ~ C(LEVEL)', data=df).fit()
anovatable = sm.stats.anova_lm(model, typ=2)

f_stat = anovatable.loc['C(LEVEL)', 'F']
p_val = anovatable.loc['C(LEVEL)', 'PR(>F)']
gmeans = df.groupby('LEVEL')['TOTALTRAINING'].mean()

print('=' * 40)
print('ANOVA (TOTALTRAINING VS LEVEL)')
print('=' * 40)
etasq = anovatable.loc['C(LEVEL)', 'sum_sq'] / anovatable['sum_sq'].sum()
print(f'Effect Size (η²): {etasq:.4f}')
pd.DataFrame({
    'METRIC': [*[f'MEAN TRAINING (LEVEL {x})' for x in gmeans.index], 'F-STATISTIC', 'P-VALUE'],
    'VALUE': [*gmeans.values, f_stat, p_val]
})

ANOVA (TOTALTRAINING VS LEVEL)
Effect Size (η²): 0.2032


,METRIC,VALUE
0,MEAN TRAINING (LEVEL 1),0.6538
1,MEAN TRAINING (LEVEL 2),0.8154
2,MEAN TRAINING (LEVEL 3),0.8587
3,MEAN TRAINING (LEVEL 4),1.0970
4,MEAN TRAINING (LEVEL 5),1.1417
5,MEAN TRAINING (LEVEL 6),1.9469
6,MEAN TRAINING (LEVEL 7),2.3226
7,MEAN TRAINING (LEVEL 8),5.2036
8,MEAN TRAINING (LEVEL 10),7.3556
9,MEAN TRAINING (LEVEL 11),6.6571


___
Structural Relationships (Categorical)

In [9]:
# 3.1 TYPE VS TRAINED (CHI-SQUARE)
ctable = pd.crosstab(df['TYPE'], df['STATUS'])
chi2, p_val, dof, expected = stats.chi2_contingency(ctable)

n = ctable.values.sum()
cramersv = np.sqrt(chi2 / (n * (min(ctable.shape) - 1))) # EFFECT SIZE: CRAMER'S V
prop = ctable.div(ctable.sum(axis=1), axis=0)

metrics = []
values = []
for i in prop.index:
    metrics.append(f'P(STATUS=1 | {i})')
    values.append(prop.loc[i, 1] if 1 in prop.columns else 0)

metrics += ['CHI2', 'P-VALUE', "CRAMER'S V"]
values += [chi2, p_val, cramersv]

print('=' * 40)
print('CHI-SQUARE (TYPE VS TRAINED)')
print('=' * 40)
pd.DataFrame({
    'METRIC': metrics,
    'VALUE': values
})

CHI-SQUARE (TYPE VS TRAINED)


,METRIC,VALUE
0,P(STATUS=1 | Governance & Support),0.8946
1,P(STATUS=1 | Infrastructure & Rolling Stock),0.5957
2,P(STATUS=1 | Operations),0.5902
3,P(STATUS=1 | Strategic Business & Real Estate),0.9846
4,CHI2,244.4040
5,P-VALUE,0.0000
6,CRAMER'S V,0.1719


In [10]:
# 3.2 TYPE VS TRAINED (CHI-SQUARE)
ctable = pd.crosstab(df['DEPARTMENT'], df['STATUS'])
chi2, p_val, dof, expected = stats.chi2_contingency(ctable)

n = ctable.values.sum()
cramersv = np.sqrt(chi2 / (n * (min(ctable.shape) - 1))) # EFFECT SIZE: CRAMER'S V
prop = ctable.div(ctable.sum(axis=1), axis=0)

metrics = []
values = []
for i in prop.index:
    metrics.append(f'P(STATUS=1 | {i})')
    values.append(prop.loc[i, 1] if 1 in prop.columns else 0)

metrics += ['CHI2', 'P-VALUE', "CRAMER'S V"]
values += [chi2, p_val, cramersv]

print('=' * 40)
print('CHI-SQUARE (DEPARTMENT VS TRAINED)')
print('=' * 40)
pd.DataFrame({
    'METRIC': metrics,
    'VALUE': values
})

CHI-SQUARE (DEPARTMENT VS TRAINED)


,METRIC,VALUE
0,P(STATUS=1 | Board of Commissioners Coordinati...,1.0000
1,P(STATUS=1 | Civil Engineering Department),0.7778
2,P(STATUS=1 | Electrified Rail Management Bureau),1.0000
3,P(STATUS=1 | Finance and Accouting Department),0.8415
4,P(STATUS=1 | Freight Service Department),0.6094
5,P(STATUS=1 | Governor Bureau),0.8444
6,P(STATUS=1 | Human Resource Department),0.8588
7,P(STATUS=1 | Information Technology Department),0.8333
8,P(STATUS=1 | Institute of Railway Training),1.0000
9,P(STATUS=1 | Internal Auditing Department),1.0000


___
Correlation Analysis

In [11]:
# 4.1 CORRELATION: AGE VS TRAINING
pearson_corr, p_val = stats.pearsonr(df['AGE'], df['TOTALTRAINING'])
spearman_corr, sp_p = stats.spearmanr(df['AGE'], df['TOTALTRAINING'])
print('=' * 40)
print('CORRELATION (AGE VS TOTALTRAINING)')
print('=' * 40)
pd.DataFrame({
    'METRIC': ['PEARSON R', 'P-VALUE (PEARSON)', 'SPEARMAN RHO', 'P-VALUE (SPEARMAN)'],
    'VALUE': [pearson_corr, p_val, spearman_corr, sp_p]
})

CORRELATION (AGE VS TOTALTRAINING)


,METRIC,VALUE
0,PEARSON R,-0.1807
1,P-VALUE (PEARSON),0.0000
2,SPEARMAN RHO,-0.2262
3,P-VALUE (SPEARMAN),0.0000


In [12]:
# 4.2 CORRELATION: RETIREMENT VS TRAINING
pearson_corr, p_val = stats.pearsonr(df['YEARS2RETIREMENT'], df['TOTALTRAINING'])
spearman_corr, sp_p = stats.spearmanr(df['YEARS2RETIREMENT'], df['TOTALTRAINING'])
print('=' * 50)
print('CORRELATION (YEARS TO RETIREMENT VS TOTALTRAINING)')
print('=' * 50)
pd.DataFrame({
    'METRIC': ['PEARSON R', 'P-VALUE (PEARSON)', 'SPEARMAN RHO', 'P-VALUE (SPEARMAN)'],
    'VALUE': [pearson_corr, p_val, spearman_corr, sp_p]
})

CORRELATION (YEARS TO RETIREMENT VS TOTALTRAINING)


,METRIC,VALUE
0,PEARSON R,0.1812
1,P-VALUE (PEARSON),0.0000
2,SPEARMAN RHO,0.2264
3,P-VALUE (SPEARMAN),0.0000


___
What Drives Training Participation?

In [13]:
# 5.1 LOGISTIC REGRESSION
lr = smf.logit('STATUS ~ AGE + C(TYPE) + C(LEVEL)', data=df).fit()
llf = lr.llf
llnull = lr.llnull
pseudo_r2 = 1 - (llf / llnull)

print('=' * 40)
print('LOGISTIC REGRESSION (TRAINED)')
print('=' * 40)
pd.DataFrame({
    'METRIC': ['LOG-LIKELIHOOD', 'NULL LOG-LIKELIHOOD', 'PSEUDO R-SQUARED', 'N OBSERVATIONS'],
    'VALUE': [llf, llnull, pseudo_r2, int(model.nobs)]
})

Optimization terminated successfully.
         Current function value: 0.592985
         Iterations 8
LOGISTIC REGRESSION (TRAINED)


,METRIC,VALUE
0,LOG-LIKELIHOOD,-4905.7642
1,NULL LOG-LIKELIHOOD,-5503.6156
2,PSEUDO R-SQUARED,0.1086
3,N OBSERVATIONS,8273.0000


In [14]:
# 5.2 SIGNIFICANT ODDS RATIOS
params = lr.params
conf = lr.conf_int()

oddsratios = pd.DataFrame({
    'METRIC': params.index,
    'OR': np.exp(params),
    'LOWER CI': np.exp(conf[0]),
    'UPPER CI': np.exp(conf[1]),
    'P-VALUE': model.pvalues
}).reset_index(drop=True)

print('=' * 40)
print('SIGNIFICANT ODDS RATIOS (P < 0.05)')
print('=' * 40)
oddsratios[oddsratios['P-VALUE'] < 0.05].sort_values(by='OR', ascending=False)

SIGNIFICANT ODDS RATIOS (P < 0.05)


,METRIC,OR,LOWER CI,UPPER CI,P-VALUE
3,C(TYPE)[T.Strategic Business & Real Estate],17.8460,3.7467,85.0026,0.0000
1,C(TYPE)[T.Infrastructure & Rolling Stock],15.5664,6.8571,35.3372,0.0000
2,C(TYPE)[T.Operations],15.3900,4.0598,58.3413,0.0000
12,C(LEVEL)[T.11],12.8608,6.6166,24.9976,0.0000
4,C(LEVEL)[T.2],7.4331,1.4272,38.7119,0.0010
11,C(LEVEL)[T.10],4.2866,2.2843,8.0442,0.0000
10,C(LEVEL)[T.8],2.1539,1.1898,3.8991,0.0007


___
Do Clusters Differ Meaningfully? (Validating results from unsupervised learning K-Means clustering)

In [15]:
# 6.0 BUILDING K-MEANS CLUSTERS (x_x)
# Note: AGE & TYPE HAS BEEN EXCLUDED HERE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

f = ['LEVEL', 'GENERAL', 'TECHNICAL', 'DIGITAL', 'TOTALTRAINING']
X = df[f]
scaler = StandardScaler()
scdata = scaler.fit_transform(X)

km6 = KMeans(n_clusters=6, n_init=10, random_state=1234)
df['CLUSTER'] = km6.fit_predict(scdata)

In [16]:
# 6.1 CLUSTER VALIDATION
f = ['LEVEL', 'GENERAL', 'TECHNICAL', 'DIGITAL', 'TOTALTRAINING']
for var in f:
    model = smf.ols(f'{var} ~ C(CLUSTER)', data=df).fit()
    anovatable = sm.stats.anova_lm(model, typ=2)

    f_stat = anovatable.loc['C(CLUSTER)', 'F']
    p_val = anovatable.loc['C(CLUSTER)', 'PR(>F)']
    etasq = anovatable.loc['C(CLUSTER)', 'sum_sq'] / anovatable['sum_sq'].sum()
    gmeans = df.groupby('CLUSTER')[var].mean()

    print('=' * 40)
    print(f'ANOVA ({var} VS CLUSTER)')
    print('=' * 40)
    print(f'Effect Size (η²): {etasq:.4f}')

    display(pd.DataFrame({
        'METRIC': [*[f'MEAN {var} (CLUSTER {x})' for x in gmeans.index], 'F-STATISTIC','P-VALUE'],
        'VALUE': [*gmeans.values, f_stat, p_val]
    }))

ANOVA (LEVEL VS CLUSTER)
Effect Size (η²): 0.6193


,METRIC,VALUE
0,MEAN LEVEL (CLUSTER 0),5.9053
1,MEAN LEVEL (CLUSTER 1),6.7398
2,MEAN LEVEL (CLUSTER 2),2.8626
3,MEAN LEVEL (CLUSTER 3),6.6172
4,MEAN LEVEL (CLUSTER 4),6.3382
5,MEAN LEVEL (CLUSTER 5),8.3988
6,F-STATISTIC,2690.0322
7,P-VALUE,0.0000


ANOVA (GENERAL VS CLUSTER)
Effect Size (η²): 0.7124


,METRIC,VALUE
0,MEAN GENERAL (CLUSTER 0),0.1023
1,MEAN GENERAL (CLUSTER 1),1.1789
2,MEAN GENERAL (CLUSTER 2),0.1115
3,MEAN GENERAL (CLUSTER 3),3.3445
4,MEAN GENERAL (CLUSTER 4),0.7887
5,MEAN GENERAL (CLUSTER 5),10.5434
6,F-STATISTIC,4095.5700
7,P-VALUE,0.0000


ANOVA (TECHNICAL VS CLUSTER)
Effect Size (η²): 0.6797


,METRIC,VALUE
0,MEAN TECHNICAL (CLUSTER 0),0.3834
1,MEAN TECHNICAL (CLUSTER 1),7.9146
2,MEAN TECHNICAL (CLUSTER 2),0.6240
3,MEAN TECHNICAL (CLUSTER 3),1.9569
4,MEAN TECHNICAL (CLUSTER 4),2.4030
5,MEAN TECHNICAL (CLUSTER 5),1.6590
6,F-STATISTIC,3509.1170
7,P-VALUE,0.0000


ANOVA (DIGITAL VS CLUSTER)
Effect Size (η²): 0.6356


,METRIC,VALUE
0,MEAN DIGITAL (CLUSTER 0),0.0297
1,MEAN DIGITAL (CLUSTER 1),0.2439
2,MEAN DIGITAL (CLUSTER 2),0.0132
3,MEAN DIGITAL (CLUSTER 3),3.0431
4,MEAN DIGITAL (CLUSTER 4),0.1213
5,MEAN DIGITAL (CLUSTER 5),1.6532
6,F-STATISTIC,2883.6614
7,P-VALUE,0.0000


ANOVA (TOTALTRAINING VS CLUSTER)
Effect Size (η²): 0.7866


,METRIC,VALUE
0,MEAN TOTALTRAINING (CLUSTER 0),0.5154
1,MEAN TOTALTRAINING (CLUSTER 1),9.3374
2,MEAN TOTALTRAINING (CLUSTER 2),0.7487
3,MEAN TOTALTRAINING (CLUSTER 3),8.3445
4,MEAN TOTALTRAINING (CLUSTER 4),3.3130
5,MEAN TOTALTRAINING (CLUSTER 5),13.8555
6,F-STATISTIC,6094.1833
7,P-VALUE,0.0000


In [17]:
# 6.2 TUKEY HSD (TOTALTRAINING)
tukey = pairwise_tukeyhsd(endog=df['TOTALTRAINING'], groups=df['CLUSTER'], alpha=0.05)
print('=' * 40)
print('TUKEY HSD (TOTALTRAINING VS CLUSTER)')
print('=' * 40)
pd.DataFrame(data=tukey.summary().data[1:], columns=tukey.summary().data[0])

TUKEY HSD (TOTALTRAINING VS CLUSTER)


,group1,group2,meandiff,p-adj,lower,upper,reject
0,0,1,8.8220,0.0000,8.5576,9.0865,True
1,0,2,0.2333,0.0000,0.1232,0.3434,True
2,0,3,7.8291,0.0000,7.5435,8.1147,True
3,0,4,2.7976,0.0000,2.6824,2.9129,True
4,0,5,13.3401,0.0000,13.0276,13.6526,True
5,1,2,-8.5887,0.0000,-8.8599,-8.3175,True
6,1,3,-0.9929,0.0000,-1.3710,-0.6148,True
7,1,4,-6.0244,0.0000,-6.2977,-5.7510,True
8,1,5,4.5181,0.0000,4.1193,4.9169,True
9,2,3,7.5958,0.0000,7.3040,7.8877,True


___
Retirement Risk Differences (Retirement versus training)

In [18]:
# 7.1 RETIREMENT RISK CATEGORY
def riskband(x):
    if x <= 5:
        return 'HIGH'
    elif x <= 10:
        return 'MEDIUM'
    else:
        return 'LOW'
df['RISK'] = df['YEARS2RETIREMENT'].apply(riskband)

rcounts = df['RISK'].value_counts()
print('=' * 30)
print('RETIREMENT RISK DISTRIBUTION')
print('=' * 30)
pd.DataFrame({
    'METRIC': [f'COUNT ({x})' for x in rcounts.index],
    'VALUE': rcounts.values
})

# THIS IS THE SAME AS '9. RISK COVERAGE' DONE IN WORKFLOW4B UNDER UNSUPERVISED LEARNING - RETIREMENT DYNAMICS

RETIREMENT RISK DISTRIBUTION


,METRIC,VALUE
0,COUNT (LOW),5258
1,COUNT (HIGH),1530
2,COUNT (MEDIUM),1485


In [19]:
# 7.1.1 RETIREMENT RISK CATEGORY (FISCAL POLICY SHIFT) - NORMALISED
# INSIGHT: THE POLICY IS REDISTRIBUTING RETIREMENT PRESSURE OVER TIME
x = df.groupby('RETIREGROUP')['RISK'].value_counts(normalize=True) * 100
pd.DataFrame(x)

proportion
RETIREGROUP RISK              
A (JAN-SEP) LOW        62.9423
            HIGH       18.8237
            MEDIUM     18.2339
B (OCT-DEC) LOW        65.2835
            HIGH       17.5657
            MEDIUM     17.1508

In [20]:
# 7.2 TRAINING VS RISK (ANOVA)
model = smf.ols('TOTALTRAINING ~ C(RISK)', data=df).fit()
anovatable = sm.stats.anova_lm(model, typ=2)

f_stat = anovatable.loc['C(RISK)', 'F']
p_val = anovatable.loc['C(RISK)', 'PR(>F)']
gmeans = df.groupby('RISK')['TOTALTRAINING'].mean()

print('=' * 35)
print('ANOVA (TOTALTRAINING VS RISK)')
print('=' * 35)
etasq = anovatable.loc['C(RISK)', 'sum_sq'] / anovatable['sum_sq'].sum()
print(f'Effect Size (η²): {etasq:.4f}')

pd.DataFrame({
    'METRIC': [*[f'MEAN TRAINING ({x})' for x in gmeans.index], 'F-STATISTIC', 'P-VALUE'],
    'VALUE': [*gmeans.values, f_stat, p_val]
})

ANOVA (TOTALTRAINING VS RISK)
Effect Size (η²): 0.0195


,METRIC,VALUE
0,MEAN TRAINING (HIGH),1.1176
1,MEAN TRAINING (LOW),2.2115
2,MEAN TRAINING (MEDIUM),1.6997
3,F-STATISTIC,82.3209
4,P-VALUE,0.0000


In [21]:
# 7.3 RISK VS TYPE (CHI-SQUARE)
ctable = pd.crosstab(df['TYPE'], df['RISK'])
chi2, p_val, dof, expected = stats.chi2_contingency(ctable)

n = ctable.values.sum()
cramersv = np.sqrt(chi2 / (n * (min(ctable.shape) - 1)))
prop = ctable.div(ctable.sum(axis=1), axis=0)

metrics = []
values = []
for i in prop.index:
    for col in prop.columns:
        metrics.append(f'P(RISK={col} | {i})')
        values.append(prop.loc[i, col])

metrics += ['CHI2', 'P-VALUE', "CRAMER'S V"]
values += [chi2, p_val, cramersv]

print('=' * 55)
print('CHI-SQUARE (TYPE VS RISK)')
print('=' * 55)
pd.DataFrame({
    'METRIC': metrics,
    'VALUE': values
})

CHI-SQUARE (TYPE VS RISK)


,METRIC,VALUE
0,P(RISK=HIGH | Governance & Support),0.1606
1,P(RISK=LOW | Governance & Support),0.6097
2,P(RISK=MEDIUM | Governance & Support),0.2297
3,P(RISK=HIGH | Infrastructure & Rolling Stock),0.1756
4,P(RISK=LOW | Infrastructure & Rolling Stock),0.6708
5,P(RISK=MEDIUM | Infrastructure & Rolling Stock),0.1536
6,P(RISK=HIGH | Operations),0.2037
7,P(RISK=LOW | Operations),0.5917
8,P(RISK=MEDIUM | Operations),0.2046
9,P(RISK=HIGH | Strategic Business & Real Estate),0.1077


In [22]:
# 8.1 CAPABILITY LOSS TIMING SHIFT
# A question to think about: Does below tell us whether the fiscal policy is smoothing loss or postponing a larger shock?
df.groupby(['RETIREMENTYEAR', 'RETIREGROUP'])['TOTALTRAINING'].sum().unstack(fill_value=0)

RETIREGROUP,A (JAN-SEP),B (OCT-DEC)
RETIREMENTYEAR,,
2025,2,1
2026,289,58
2027,172,123
2028,243,62
2029,348,103
2030,207,102
2031,332,78
2032,307,76
2033,356,90


___
**END**